In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset, DataLoader
import h5py
from typing import Optional, Tuple

class SAE(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.encode = nn.Linear(input_size, hidden_size)
        self.decode = nn.Linear(hidden_size, input_size)
        
    def forward(self, x):
        features = self.encode(x)
        features = F.relu(features)
        reconstruction = self.decode(features)
        return reconstruction, features
    
    def get_decoder_norms(self):
        return torch.linalg.vector_norm(self.decode.weight, dim=0)


class ResidualStreamDataset(Dataset):
    def __init__(self, path: str, num_examples: Optional[int] = None):
        self.file = h5py.File(path, 'r')
        self.activations = self.file['activations']
        if num_examples is not None:
            self.activations = self.activations[:num_examples]
        
    def __len__(self):
        return len(self.activations)
    
    def __getitem__(self, idx):
        return torch.from_numpy(self.activations[idx]).view(torch.bfloat16)
    
    def __del__(self):
        self.file.close()

def get_dataloader(
    path: str,
    batch_size: int,
    num_examples: Optional[int] = None,
    num_workers: int = 4,
    shuffle: bool = True
) -> DataLoader:
    dataset = ResidualStreamDataset(path, num_examples)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True
    )

def train_sae(
    data_path: str,
    input_size: int = 2048,
    hidden_size: int = 8192,
    batch_size: int = 32,
    num_epochs: int = 1000,
    lr: float = 1e-3,
    lambda_max: float = 5.0,
    warmup_fraction: float = 0.2,
    num_workers: int = 4,
    num_batches_to_overfit: int = 10  # New parameter
):
    # Setup data
    train_loader = get_dataloader(
        data_path,
        batch_size=batch_size,
        num_workers=num_workers
    )
    
    # Get the batches we want to overfit
    overfit_batches = []
    for i, batch in enumerate(train_loader):
        if i >= num_batches_to_overfit:
            break
        overfit_batches.append(batch.cuda())
    
    # Initialize model
    model = SAE(input_size, hidden_size).cuda().to(torch.bfloat16)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Training loop
    lambda_l1 = 0  # Start at 0 and increase during warmup
    warmup_epochs = int(num_epochs * warmup_fraction)
    
    from tqdm import tqdm
    for epoch in tqdm(range(num_epochs)):
        total_loss = 0
        total_mse = 0
        total_l1 = 0
        
        # Train on our few fixed batches
        for batch in overfit_batches:
            reconstruction, features = model(batch)
            
            # Losses
            mse_loss = F.mse_loss(batch, reconstruction)
            decoder_norms = model.get_decoder_norms()
            l1_loss = torch.sum(torch.abs(features) * decoder_norms[None, :]) / (batch.shape[0] * features.shape[1])
            loss = mse_loss + (lambda_l1 * l1_loss)
            
            # Optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Track metrics
            total_loss += loss.item()
            total_mse += mse_loss.item()
            total_l1 += l1_loss.item()
        
        # Update lambda during warmup
        if epoch < warmup_epochs:
            lambda_l1 = (epoch / warmup_epochs) * lambda_max
            
        # Log metrics
        if epoch % 100 == 0:
            avg_loss = total_loss / len(overfit_batches)
            avg_mse = total_mse / len(overfit_batches)
            avg_l1 = total_l1 / len(overfit_batches)
            
            with torch.no_grad():
                # Get metrics from last batch
                mean_activation = features.abs().mean().item()
                sparsity = (features > 0).float().mean().item()
                active_features = torch.linalg.vector_norm(features, ord=0, dim=0).sum().item()/batch_size
                recon_error = torch.linalg.vector_norm(reconstruction - batch) / torch.linalg.vector_norm(batch)
            
            print(f"Epoch {epoch}, Total Loss: {avg_loss:.6f}")
            print(f"- MSE Loss: {avg_mse:.6f}")
            print(f"- L1 Loss: {avg_l1:.6f}")
            print(f"- Mean activation: {mean_activation:.4f}")
            print(f"- Sparsity: {sparsity:.4f}")
            print(f"- Active features: {active_features:.1f}")
            print(f"- Reconstruction error: {recon_error:.4f}")
            print("---")
    
    return model

if __name__ == "__main__":
    # Training settings
    settings = {
        "data_path": '/home/henry/Documents/PythonProjects/open-concept-steering-dataset/residual_stream_activations_llama1b_bf16.h5',
        "input_size": 2048,
        "hidden_size": 8192*4,
        "batch_size": 32,
        "num_epochs": 1000,
        "lr": 1e-3,
        "lambda_max": 5.0,
        "warmup_fraction": 0.2,
        "num_workers": 24
    }
    
    model = train_sae(**settings)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset, DataLoader
import h5py
from typing import Optional, Tuple

class SAE(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.encode = nn.Linear(input_size, hidden_size)
        self.decode = nn.Linear(hidden_size, input_size)
        
    def forward(self, x):
        features = self.encode(x)
        features = F.relu(features)
        reconstruction = self.decode(features)
        return reconstruction, features
    
    def get_decoder_norms(self):
        return torch.linalg.vector_norm(self.decode.weight, dim=0)


class ResidualStreamDataset(Dataset):
    def __init__(self, path: str, num_examples: Optional[int] = None):
        self.file = h5py.File(path, 'r')
        self.activations = self.file['activations']
        if num_examples is not None:
            self.activations = self.activations[:num_examples]
        
    def __len__(self):
        return len(self.activations)
    
    def __getitem__(self, idx):
        return torch.from_numpy(self.activations[idx]).view(torch.bfloat16).to(torch.float32) #load from uint16 to bfloat16, then cast to float32. there must be a better way
    
    def __del__(self):
        self.file.close()

def get_dataloader(
    path: str,
    batch_size: int,
    num_examples: Optional[int] = None,
    num_workers: int = 4,
    shuffle: bool = True
) -> DataLoader:
    dataset = ResidualStreamDataset(path, num_examples)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True
    )

def train_sae(
    data_path: str,
    input_size: int = 2048,
    hidden_size: int = 8192,
    batch_size: int = 32,
    num_epochs: int = 1000,
    lr: float = 1e-3,
    lambda_max: float = 5.0,
    warmup_fraction: float = 0.2,
    num_workers: int = 4,
    num_batches_to_overfit: int = 10
):
    # Setup data
    train_loader = get_dataloader(
        data_path,
        batch_size=batch_size,
        num_workers=num_workers
    )
    
    # Get the batches we want to overfit
    overfit_batches = []
    for i, batch in enumerate(train_loader):
        if i >= num_batches_to_overfit:
            break
        overfit_batches.append(batch.cuda())
    
    # Initialize model
    model = SAE(input_size, hidden_size).cuda()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Training loop
    lambda_l1 = 0
    warmup_epochs = int(num_epochs * warmup_fraction)
    
    from tqdm import tqdm
    for epoch in tqdm(range(num_epochs)):
        total_loss = 0
        total_mse = 0
        total_l1 = 0
        
        # Train on our fixed batches
        for batch in overfit_batches:
            # Use autocast for mixed precision
            with torch.autocast("cuda", dtype=torch.bfloat16):
                reconstruction, features = model(batch)
                
                # Losses
                mse_loss = F.mse_loss(batch, reconstruction)
                decoder_norms = model.get_decoder_norms()
                l1_loss = torch.sum(torch.abs(features) * decoder_norms[None, :]) / (batch.shape[0] * features.shape[1])
                loss = mse_loss + (lambda_l1 * l1_loss)
            
            # Optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Track metrics
            total_loss += loss.item()
            total_mse += mse_loss.item()
            total_l1 += l1_loss.item()
        
        # Update lambda during warmup
        if epoch < warmup_epochs:
            lambda_l1 = (epoch / warmup_epochs) * lambda_max
            
        # Log metrics
        if epoch % 100 == 0:
            avg_loss = total_loss / len(overfit_batches)
            avg_mse = total_mse / len(overfit_batches)
            avg_l1 = total_l1 / len(overfit_batches)
            
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
                # Get metrics from last batch
                mean_activation = features.abs().mean().item()
                sparsity = (features > 0).float().mean().item()
                active_features = torch.linalg.vector_norm(features, ord=0, dim=0).sum().item()/batch_size
                recon_error = torch.linalg.vector_norm(reconstruction - batch) / torch.linalg.vector_norm(batch)
            
            print(f"Epoch {epoch}, Total Loss: {avg_loss:.6f}")
            print(f"- MSE Loss: {avg_mse:.6f}")
            print(f"- L1 Loss: {avg_l1:.6f}")
            print(f"- Mean activation: {mean_activation:.4f}")
            print(f"- Sparsity: {sparsity:.4f}")
            print(f"- Active features: {active_features:.1f}")
            print(f"- Reconstruction error: {recon_error:.4f}")
            print("---")
    
    return model

if __name__ == "__main__":
    # Training settings
    settings = {
        "data_path": '/home/henry/Documents/PythonProjects/open-concept-steering-dataset/residual_stream_activations_llama1b_bf16.h5',
        "input_size": 2048,
        "hidden_size": 8192*4,
        "batch_size": 32,
        "num_epochs": 1000,
        "lr": 1e-3,
        "lambda_max": 5.0,
        "warmup_fraction": 0.2,
        "num_workers": 48
    }
    
    model = train_sae(**settings)

  0%|          | 2/1000 [00:00<06:13,  2.67it/s]

Epoch 0, Total Loss: 51.287574
- MSE Loss: 51.287574
- L1 Loss: 0.007303
- Mean activation: 0.0312
- Sparsity: 0.3893
- Active features: 12756.1
- Reconstruction error: 18.1872
---


 10%|█         | 102/1000 [00:18<02:41,  5.55it/s]

Epoch 100, Total Loss: 0.011492
- MSE Loss: 0.008546
- L1 Loss: 0.001190
- Mean activation: 0.0025
- Sparsity: 0.0486
- Active features: 1593.9
- Reconstruction error: 0.1252
---


 20%|██        | 202/1000 [00:36<02:23,  5.55it/s]

Epoch 200, Total Loss: 0.002453
- MSE Loss: 0.000128
- L1 Loss: 0.000467
- Mean activation: 0.0003
- Sparsity: 0.0047
- Active features: 154.9
- Reconstruction error: 0.0423
---


 22%|██▏       | 222/1000 [00:40<02:20,  5.54it/s]